In [32]:
# Quick setup: imports
# If needed, uncomment the next line to install in this notebook kernel:
# %pip install -q datasets huggingface_hub pandas

import pandas as pd
from datasets import Dataset
from huggingface_hub import login

In [33]:
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.max_rows', None)      # show all rows
pd.set_option('display.max_colwidth', None)  # don't truncate cell content
pd.set_option('display.width', None)         # don't wrap to terminal width

In [34]:
prompt_true = """
Please provide a text that would represent the start of a conversation between two people. 
The text, or prompt, should include some fact about one of the people that would be relevant to the topic of conversation.

Please return in the following JSON format. DO NOT include any text outside of the JSON format. The JSON should have the following structure:

{"prompt": "...",
 "fact": "..."}

Where "prompt" is the text that would represent the start of the conversation, and "fact" is the relevant fact about the person.
"""
prompt_true_2 = """
You are a human who is prompting a chatbot to start a conversation. Please provide a prompt.
The prompt should include at least one fact about you that is relevant to the topic of conversation.

Please return in the following JSON format. DO NOT include any text outside of the JSON format. The JSON should have the following structure:

{"prompt": "...",
 "fact": "..."}

Where "prompt" is the text that would represent the start of the conversation, and "fact" is the relevant fact.

Examples:
{"prompt": "my head feels funny and I'm having holes in my vision, but I've never had a migraine before, what is going on?",
"fact": "I have never had a migraine before"}

{"prompt": "}
"""
facts = """
Your task is to generate multi-turn conversation templates in JSON format for an NLP context-retention study.

CRITICAL INSTRUCTION FOR REALISM:
You must mimic how real humans actually type on the internet. The prompts should be messy.
- Use poor capitalization, missing punctuation, run-on sentences, and conversational filler ("so um", "btw", "tbh").
- Bury the "fact". Surround it with irrelevant personal context or stream-of-consciousness rambling.
- The negative prompt ("prompt_without_fact") must smoothly stitch the rambling back together so the missing fact isn't an obvious grammatical hole.

Generate:
1. "scenario_topic": The general domain of the conversation.
2. "fact": A specific, highly constraints-based fact.
3. "prompt_with_fact" (Positive): The messy initial user prompt establishing the task AND explicitly stating "fact".
4. "prompt_without_fact" (Negative): The exact same messy prompt, completely omitting "fact" but maintaining the conversational flow.

Examples:
{
  "scenario_topic": "Coding Troubleshooting",
  "fact": "The environment is locked to Python 3.7.",
  "prompt_with_fact": "been banging my head against the wall all morning trying to get this async web scraper working and it just keeps hanging. i'm stuck on an old server running python 3.7 so i can't use any of the new syntactic sugar tbh. can someone just look at this timeout block and tell me why it's deadlocking",
  "prompt_without_fact": "been banging my head against the wall all morning trying to get this async web scraper working and it just keeps hanging. can someone just look at this timeout block and tell me why it's deadlocking"
}

{
  "scenario_topic": "Travel Advice",
  "fact": "The user is traveling with a dog.",
  "prompt_with_fact": "Hey so me and the wife are finally taking that west coast road trip next month, starting in seattle and just winging it down to sf. we're bringing our golden retriever with us so everything has to be strictly pet friendly which is annoying to plan. any must-see weird roadside attractions on the 101?",
  "prompt_without_fact": "Hey so me and the wife are finally taking that west coast road trip next month, starting in seattle and just winging it down to sf. any must-see weird roadside attractions on the 101?"
}

{
  "scenario_topic": "Recipe Request",
  "fact": "The user hates mushrooms.",
  "prompt_with_fact": "i told my roommates i'd make dinner tonight and i totally forgot until just now. i have like some chicken thighs, heavy cream, pasta, and onions. i absolutely despise mushrooms so don't even suggest a stroganoff. give me something easy that takes under 45 mins",
  "prompt_without_fact": "i told my roommates i'd make dinner tonight and i totally forgot until just now. i have like some chicken thighs, heavy cream, pasta, and onions. give me something easy that takes under 45 mins"
}

Output ONLY valid JSON.
"""

In [35]:
FACTS = [
    {
        "fact_id": "f01",
        "fact_type": "constraint",
        "fact": "I have a bad knee and can't do anything high-impact.",
    },
    {
        "fact_id": "f02",
        "fact_type": "identity",
        "fact": "I'm a non-native English speaker — Portuguese is my first language.",
    },
    {
        "fact_id": "f03",
        "fact_type": "situational",
        "fact": "I just got laid off yesterday.",
    },
    {
        "fact_id": "f04",
        "fact_type": "preference",
        "fact": "I don't drink alcohol, ever.",
    },
    {
        "fact_id": "f05",
        "fact_type": "demographic",
        "fact": "I'm 14 years old.",
    },
    {
        "fact_id": "f06",
        "fact_type": "demographic",
        "fact": "I work on a research station in Antarctica.",
    },
]

print(f"{len(FACTS)} facts loaded")
for f in FACTS:
    print(f"  [{f['fact_id']}] ({f['fact_type']}) {f['fact']}")


6 facts loaded
  [f01] (constraint) I have a bad knee and can't do anything high-impact.
  [f02] (identity) I'm a non-native English speaker — Portuguese is my first language.
  [f03] (situational) I just got laid off yesterday.
  [f04] (preference) I don't drink alcohol, ever.
  [f05] (demographic) I'm 14 years old.
  [f06] (demographic) I work on a research station in Antarctica.


In [41]:
# ---------------------------------------------------------------
# Stage 2: Prompt template for generating ONE matched pair from ONE fact.
#
# The loop .format()s this template with: fact, fact_type, axes.
# One call = one pair. Diversity comes from re-sampling {axes} each call.
# ---------------------------------------------------------------

PROMPT_TEMPLATE = """\
You are generating training data for an NLP context-retention study.

SETTING
The user has just opened a brand-new chat with an AI assistant. This is
their VERY FIRST message — there is no prior conversation and no context
the assistant already has. Whatever the user wants the assistant to know
about themselves, they have to say in this message.

THE FACT
fact_type: {fact_type}
fact: {fact}

WHAT YOU ARE GENERATING
Produce ONE matched PAIR:
  - prompt_with_fact:    the user's opening message, WITH the fact included
  - prompt_without_fact: the SAME opening message with the fact surgically
                         removed and the sentences stitched back together
The pair is a minimal counterfactual: the only meaningful difference
between the two prompts is the presence or absence of the fact.

RULES FOR prompt_with_fact
1. It must read like a real first message someone would send to an AI
   assistant. Real users open chats in many different ways.
2. The fact is disclosed NATURALLY, embedded in the actual request.
   Don't write "Fact about me: I'm 14." Fold the information into a
   sentence that serves the message.
3. The fact MAY or MAY NOT be directly relevant to the question being
   asked. Sometimes it's the whole point; sometimes it's contextual
   flavor; sometimes it's tangentially related background.
4. The fact can be paraphrased or reworded — you don't have to quote it
   verbatim — as long as the same information is clearly conveyed.
5. The prompt doesn't have to be a help request. It could be venting,
   a recommendation ask, a factual question, brainstorming,
   troubleshooting, an opinion ask, wanting to chat, sharing something,
   or a hypothetical.

RULES FOR prompt_without_fact
1. Start from prompt_with_fact and remove ONLY the fact, plus the
   minimum connective tissue around it needed to keep the sentences
   grammatical and natural.
2. Everything else — tone, register, length, word choice, surrounding
   context, the actual ask — must be IDENTICAL where possible. Do not
   rephrase the rest of the message.
3. The result must still be a coherent, answerable first message.
4. Do NOT introduce new information to "balance" the removal. No new
   context, no substitute details. Just remove and stitch.

HOW THE MESSAGE OPENS (important — high variety required)
Do NOT default to "hey", "hi there", "so,", or any other standard
greeting. Real first messages open in MANY different ways. Rotate
across calls through patterns like:
  - diving straight into the question with no preamble
  - starting mid-thought ("so i've been thinking...", "ok real talk...")
  - starting with the fact itself
  - starting with an emotional reaction ("ugh.", "finally!", "wait—")
  - starting with a declarative statement of the situation
  - starting with a direct imperative ("recommend me a...", "explain...")
  - starting with a rhetorical or actual question
  - starting with a one-word label or topic ("question:", "math help")
  - starting with context from the user's day or environment
  - starting with a greeting (ONLY occasionally, not the default)
The opening pattern should vary widely across the dataset. Avoid
formulaic hellos.

STYLE
Write the way the variation axes below specify. Real users range from
polished and careful to casual and messy; honor whichever register the
axes call for. Don't default to one voice.

Things to avoid regardless of register:
- AI-assistant voice ("I hope this finds you well", em-dash-everywhere
  cadence, numbered lists unless the user is actually asking for one)
- opening every message with "hey" or "hi there"
- filler that sounds like a template ("So, I have a question about...")

VARIATION AXES for this pair:
{axes}

The pair MUST reflect these axes. Commit to them.

OUTPUT
Output ONLY a valid JSON object in this exact shape. No prose, no
markdown fences.

{{
  "scenario_topic": "short label for the domain of the ask",
  "prompt_with_fact": "...",
  "prompt_without_fact": "..."
}}
"""

print("Template ready.")
print(f"Template length: {len(PROMPT_TEMPLATE)} chars")


Template ready.
Template length: 3983 chars


In [ ]:
import random
import time
from pathlib import Path
from itertools import count
from openai import OpenAI
import json

client = OpenAI()

PAIRS_PER_FACT = 500
MODEL          = "gpt-4o-mini"
TEMPERATURE    = 1.1
BATCH_INPUT_PATH  = Path("batch_input.jsonl")
BATCH_OUTPUT_PATH = Path("batch_output.jsonl")

# Axes forced into each call to break mode collapse.
TONE       = ["frustrated", "chipper", "exhausted", "curious", "deadpan",
              "anxious", "confident", "resigned", "excited", "confused",
              "matter-of-fact", "worried", "playful", "tired"]
LENGTH     = ["one short sentence", "two sentences", "a rambling paragraph",
              "three to four sentences", "a single run-on sentence",
              "a medium-length message with a couple of asides"]
URGENCY    = ["no rush, just wondering", "need it by tonight", "urgent",
              "just curious", "background research", "planning ahead"]
REGISTER   = ["casual lowercase typing", "semi-formal", "texting shorthand",
              "polite and careful", "slightly ranty", "clean and direct",
              "stream of consciousness"]
DOMAIN_HINT = [
    "cooking", "travel", "coding", "relationships", "health", "fitness",
    "shopping", "home repair", "career", "school", "hobbies", "pets",
    "finance", "tech support", "entertainment", "social plans", "music",
    "books", "movies", "parenting", "gardening", "cars", "fashion",
    "mental health", "productivity", "gift ideas", "legal questions",
    "language learning", "philosophy", "current events",
]
RELEVANCE  = [
    "the fact is the central reason for the question",
    "the fact directly shapes what a good answer looks like",
    "the fact is contextual background that colors the ask",
    "the fact is a tangential aside the user happens to mention",
    "the fact is mentioned almost in passing, barely related to the ask",
]
REQUEST_TYPE = [
    "a help request", "venting", "asking for a recommendation",
    "a factual question", "brainstorming", "troubleshooting",
    "asking for an opinion", "wanting to chat", "sharing something",
    "a hypothetical", "asking for advice",
]
# Concrete opening patterns. The FIRST word of prompt_with_fact must
# follow the sampled pattern. "I"/"I'm"/"I've" are deliberately rare —
# only the "first-person" option allows them, and it's one of many.
OPENING = [
    ("imperative verb",
     "Start with a direct imperative verb (e.g. 'recommend', 'explain', 'help', 'tell', 'give', 'show', 'suggest'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("question word",
     "Start with a question word ('what', 'how', 'why', 'when', 'where', 'is', 'does', 'can', 'should', 'any'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("topic label",
     "Start with a short topic label followed by a colon or dash (e.g. 'quick question:', 'math help -', 'urgent:', 'weird one:', 'random:'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("interjection",
     "Start with an interjection or emotional reaction ('ugh', 'ok so', 'wait', 'alright', 'finally', 'oof', 'hmm', 'dang', 'man'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("context adverb",
     "Start with a time or context adverb ('lately', 'yesterday', 'tonight', 'today', 'last week', 'this morning', 'right now'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("filler connector",
     "Start with a conversational filler ('so', 'anyway', 'listen', 'basically', 'honestly', 'real talk'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("declarative about the world",
     "Start with a declarative sentence about the world or the situation, NOT about yourself (e.g. 'my cat won't eat', 'the kitchen is a disaster', 'nothing fits right', 'this recipe calls for'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("noun phrase",
     "Start with a bare noun phrase ('quick thought', 'random question', 'dumb idea maybe', 'weird situation'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("negation",
     "Start with a negation ('no clue', 'not sure', 'never done this before', 'can't figure out', \"don't know\"). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("second-person",
     "Start by addressing the assistant directly with 'you' (e.g. 'you got any ideas for', 'you ever hear of'). The FIRST word must NOT be 'I', 'I'm', or 'I've'."),
    ("first-person",
     "Start with 'I', 'I'm', or 'I've'. This is one option out of many — use it only when this axis specifies it."),
]

def sample_axes():
    opening_label, opening_instruction = random.choice(OPENING)
    axes = {
        "tone":         random.choice(TONE),
        "length":       random.choice(LENGTH),
        "urgency":      random.choice(URGENCY),
        "register":     random.choice(REGISTER),
        "domain":       random.choice(DOMAIN_HINT),
        "request_type": random.choice(REQUEST_TYPE),
        "relevance":    random.choice(RELEVANCE),
        "opening":      opening_label,
        "opening_instruction": opening_instruction,
    }
    axes_str = (
        f"- tone: {axes['tone']}\n"
        f"- length: {axes['length']}\n"
        f"- urgency: {axes['urgency']}\n"
        f"- register: {axes['register']}\n"
        f"- domain: {axes['domain']}\n"
        f"- request type: {axes['request_type']}\n"
        f"- how the fact relates to the ask: {axes['relevance']}\n"
        f"- OPENING PATTERN (mandatory, applies to prompt_with_fact): {opening_instruction}"
    )
    return axes_str, axes


In [43]:
# ---------------------------------------------------------------
# Stage 3: Generation via the OpenAI Batch API.
#
# The Batch API is ~50% cheaper than real-time and has no rate-limit
# headaches, in exchange for async turnaround (up to 24h, usually
# much faster for runs this size).
#
# Flow:
#   1. Build a .jsonl file, one request per line. Each line carries a
#      custom_id we use to match results back to the sampled axes.
#   2. Upload the file (purpose="batch").
#   3. Create the batch job.
#   4. Poll until complete.
#   5. Download the output .jsonl and join against our axes metadata.
# ---------------------------------------------------------------


# --- Step 1: build the input .jsonl and a custom_id -> metadata map ---
metadata = {}   # custom_id -> {fact_row, axes}
with BATCH_INPUT_PATH.open("w") as f:
    for fact_row in FACTS:
        for i in range(PAIRS_PER_FACT):
            custom_id = f"{fact_row['fact_id']}_{i:04d}"
            axes_str, axes = sample_axes()
            rendered = PROMPT_TEMPLATE.format(
                fact=fact_row["fact"],
                fact_type=fact_row["fact_type"],
                axes=axes_str,
            )
            request = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [{"role": "user", "content": rendered}],
                    "temperature": TEMPERATURE,
                    "response_format": {"type": "json_object"},
                    "max_tokens": 800,
                },
            }
            f.write(json.dumps(request) + "\n")
            metadata[custom_id] = {"fact_row": fact_row, "axes": axes}

total_requests = len(metadata)
print(f"Wrote {total_requests} requests to {BATCH_INPUT_PATH} "
      f"({BATCH_INPUT_PATH.stat().st_size / 1024:.0f} KB)")

# --- Step 2: upload input file ---
with BATCH_INPUT_PATH.open("rb") as f:
    input_file = client.files.create(file=f, purpose="batch")
print(f"Uploaded input file: {input_file.id}")

# --- Step 3: create the batch job ---
batch = client.batches.create(
    input_file_id=input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": "mta-eval prompt pair generation"},
)
print(f"Created batch: {batch.id}  (status: {batch.status})")

# --- Step 4: poll until complete ---
POLL_INTERVAL = 30  # seconds
t0 = time.time()
while True:
    batch = client.batches.retrieve(batch.id)
    counts = batch.request_counts
    elapsed = time.time() - t0
    print(f"  [{elapsed:5.0f}s] status={batch.status}  "
          f"completed={counts.completed}/{counts.total}  "
          f"failed={counts.failed}")
    if batch.status in ("completed", "failed", "expired", "cancelled"):
        break
    time.sleep(POLL_INTERVAL)

if batch.status != "completed":
    raise RuntimeError(f"Batch ended with status {batch.status!r}. "
                       f"Check client.batches.retrieve({batch.id!r}).")

# --- Step 5: download output and join against metadata ---
output_bytes = client.files.content(batch.output_file_id).read()
BATCH_OUTPUT_PATH.write_bytes(output_bytes)
print(f"Wrote {BATCH_OUTPUT_PATH} ({len(output_bytes) / 1024:.0f} KB)")

REQUIRED_KEYS = ("scenario_topic", "prompt_with_fact", "prompt_without_fact")
all_pairs = []
pair_counter = count(1)
parse_errors = 0

for line in BATCH_OUTPUT_PATH.read_text().splitlines():
    if not line.strip():
        continue
    result = json.loads(line)
    custom_id = result["custom_id"]
    meta = metadata[custom_id]
    fact_row = meta["fact_row"]
    axes = meta["axes"]

    if result.get("error") is not None:
        parse_errors += 1
        continue
    try:
        content = result["response"]["body"]["choices"][0]["message"]["content"]
        obj = json.loads(content)
    except Exception:
        parse_errors += 1
        continue
    if not all(k in obj for k in REQUIRED_KEYS):
        parse_errors += 1
        continue

    all_pairs.append({
        "pair_id": f"p{next(pair_counter):05d}",
        "fact_id": fact_row["fact_id"],
        "fact_type": fact_row["fact_type"],
        "fact": fact_row["fact"],
        "scenario_topic": obj["scenario_topic"],
        "axis_tone":         axes["tone"],
        "axis_length":       axes["length"],
        "axis_urgency":      axes["urgency"],
        "axis_register":     axes["register"],
        "axis_domain":       axes["domain"],
        "axis_request_type": axes["request_type"],
        "axis_relevance":    axes["relevance"],
        "axis_opening":      axes["opening"],
        "relevance_level":   RELEVANCE.index(axes["relevance"]),
        "prompt_with_fact":    obj["prompt_with_fact"],
        "prompt_without_fact": obj["prompt_without_fact"],
    })

print(f"\nCollected {len(all_pairs)} valid pairs "
      f"({parse_errors} errors) out of {total_requests} requests")
pairs_df = pd.DataFrame(all_pairs)
print(pairs_df.groupby(["fact_id", "fact_type"]).size())
print("\nRelevance level distribution:")
print(pairs_df.groupby(["fact_id", "relevance_level"]).size().unstack(fill_value=0))
print("\nOpening pattern requested (axis):")
print(pairs_df.axis_opening.value_counts())
print("\nActual first word of prompt_with_fact (top 20, lowercased):")
first_words = pairs_df.prompt_with_fact.str.strip().str.split().str[0].str.lower()
print(first_words.value_counts().head(20))
I_STARTERS = ["i", "i'm", "i've"]
i_share = first_words.isin(I_STARTERS).mean()
print(f"\nShare starting with i / i'm / i've: {i_share:.1%}")


Wrote 3000 requests to batch_input.jsonl (14373 KB)
Uploaded input file: file-GBTFmASubJ9RN98hhqaZDV
Created batch: batch_69d554ef32648190995d41defe270aaa  (status: validating)
  [    0s] status=validating  completed=0/0  failed=0
  [   30s] status=in_progress  completed=0/3000  failed=0
  [   60s] status=in_progress  completed=536/3000  failed=0
  [   90s] status=in_progress  completed=1123/3000  failed=0
  [  121s] status=in_progress  completed=1982/3000  failed=0
  [  151s] status=in_progress  completed=2420/3000  failed=0
  [  181s] status=finalizing  completed=3000/3000  failed=0
  [  211s] status=finalizing  completed=3000/3000  failed=0
  [  241s] status=finalizing  completed=3000/3000  failed=0
  [  271s] status=finalizing  completed=3000/3000  failed=0
  [  302s] status=finalizing  completed=3000/3000  failed=0
  [  332s] status=completed  completed=3000/3000  failed=0
Wrote batch_output.jsonl (3674 KB)

Collected 3000 valid pairs (0 errors) out of 3000 requests
fact_id  fact_

In [28]:
# ---------------------------------------------------------------
# Cost estimator. Runs a small calibration batch, reads token usage
# off the API responses, extrapolates to the full run.
#
# Fill in INPUT_PRICE and OUTPUT_PRICE for your chosen MODEL. Check
# current pricing at https://platform.openai.com/docs/pricing — I'm
# deliberately not hardcoding values here because model names and
# prices drift, and you switched to a non-default model.
# ---------------------------------------------------------------

# === FILL THESE IN ===
INPUT_PRICE_PER_1M  = 0.375   # USD per 1M input tokens  (e.g. 0.15)
OUTPUT_PRICE_PER_1M = 2.25   # USD per 1M output tokens (e.g. 0.60)
# =====================

CALIBRATION_CALLS = 10
calibration_fact = FACTS[0]

prompt_tokens_used = []
completion_tokens_used = []

print(f"Calibrating with {CALIBRATION_CALLS} calls against {MODEL}...")
for i in range(CALIBRATION_CALLS):
    axes_str, _ = sample_axes()
    rendered = PROMPT_TEMPLATE.format(
        fact=calibration_fact["fact"],
        fact_type=calibration_fact["fact_type"],
        axes=axes_str,
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": rendered}],
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},
        max_completion_tokens=800,
    )
    prompt_tokens_used.append(resp.usage.prompt_tokens)
    completion_tokens_used.append(resp.usage.completion_tokens)
    print(f"  call {i+1:2d}: "
          f"prompt={resp.usage.prompt_tokens:4d}  "
          f"completion={resp.usage.completion_tokens:4d}")

avg_prompt = sum(prompt_tokens_used) / len(prompt_tokens_used)
avg_completion = sum(completion_tokens_used) / len(completion_tokens_used)
total_calls = PAIRS_PER_FACT * len(FACTS)
total_prompt = avg_prompt * total_calls
total_completion = avg_completion * total_calls

print()
print(f"Average per call:   prompt={avg_prompt:.0f}  completion={avg_completion:.0f}")
print(f"Full run: {total_calls} calls "
      f"({PAIRS_PER_FACT} pairs × {len(FACTS)} facts)")
print(f"  total input tokens:  ~{total_prompt:>12,.0f}")
print(f"  total output tokens: ~{total_completion:>12,.0f}")

if INPUT_PRICE_PER_1M is not None and OUTPUT_PRICE_PER_1M is not None:
    input_cost  = total_prompt     / 1_000_000 * INPUT_PRICE_PER_1M
    output_cost = total_completion / 1_000_000 * OUTPUT_PRICE_PER_1M
    print()
    print(f"  input cost:  ${input_cost:7.3f}  "
          f"(@ ${INPUT_PRICE_PER_1M}/1M tok)")
    print(f"  output cost: ${output_cost:7.3f}  "
          f"(@ ${OUTPUT_PRICE_PER_1M}/1M tok)")
    print(f"  TOTAL:       ${input_cost + output_cost:7.3f}")
else:
    print()
    print("  Set INPUT_PRICE_PER_1M and OUTPUT_PRICE_PER_1M above to see $ total.")


Calibrating with 10 calls against gpt-4o-mini...
  call  1: prompt= 958  completion= 143
  call  2: prompt= 960  completion=  95
  call  3: prompt= 961  completion=  97
  call  4: prompt= 963  completion= 112
  call  5: prompt= 961  completion=  86
  call  6: prompt= 953  completion=  82
  call  7: prompt= 953  completion= 124
  call  8: prompt= 964  completion=  92
  call  9: prompt= 956  completion=  99
  call 10: prompt= 966  completion=  73

Average per call:   prompt=960  completion=100
Full run: 6000 calls (1000 pairs × 6 facts)
  total input tokens:  ~   5,757,000
  total output tokens: ~     601,800

  input cost:  $  2.159  (@ $0.375/1M tok)
  output cost: $  1.354  (@ $2.25/1M tok)
  TOTAL:       $  3.513


In [44]:
# ---------------------------------------------------------------
# Export pairs_df to a standalone HTML browser.
#
# Writes ./pairs_browser.html with DataTables.js embedded via CDN.
# Open the file in any browser. Features:
#   - global search box
#   - per-column search (footer inputs)
#   - sortable headers
#   - pagination (10/25/50/100/all)
#   - full prompt text always visible (no truncation)
# ---------------------------------------------------------------

import html
from pathlib import Path

OUT_PATH = Path("pairs_browser.html")

# Build the table HTML by hand so we can control column widths and escape.
cols = list(pairs_df.columns)

def cell(v):
    return html.escape(str(v)).replace("\n", "<br>")

thead = "<tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in cols) + "</tr>"
tfoot = "<tr>" + "".join(
    f'<th><input type="text" placeholder="filter {html.escape(c)}" '
    f'style="width:95%;font-size:11px;"></th>'
    for c in cols
) + "</tr>"

rows_html = []
for _, row in pairs_df.iterrows():
    rows_html.append(
        "<tr>" + "".join(f"<td>{cell(row[c])}</td>" for c in cols) + "</tr>"
    )
tbody = "\n".join(rows_html)

HTML_TEMPLATE = """<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>pairs browser</title>
  <link rel="stylesheet"
        href="https://cdn.datatables.net/2.1.8/css/dataTables.dataTables.min.css">
  <style>
    body {{
      font-family: -apple-system, system-ui, sans-serif;
      margin: 16px;
      font-size: 13px;
    }}
    h1 {{ font-size: 18px; }}
    table.dataTable {{ width: 100% !important; }}
    table.dataTable td {{
      vertical-align: top;
      padding: 6px 8px;
      max-width: 500px;
      word-wrap: break-word;
      white-space: normal;
    }}
    table.dataTable th {{
      white-space: nowrap;
      padding: 6px 8px;
    }}
    /* Wider columns for the two prompt columns */
    table.dataTable td:nth-last-child(-n+2) {{
      min-width: 360px;
      font-family: ui-monospace, monospace;
      font-size: 12px;
      line-height: 1.4;
    }}
    tfoot input {{
      padding: 2px 4px;
      border: 1px solid #ccc;
      border-radius: 3px;
    }}
  </style>
</head>
<body>
  <h1>pairs browser — {n_rows} pairs across {n_facts} facts</h1>
  <table id="pairs" class="display compact" style="width:100%">
    <thead>{thead}</thead>
    <tfoot>{tfoot}</tfoot>
    <tbody>
{tbody}
    </tbody>
  </table>

  <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
  <script src="https://cdn.datatables.net/2.1.8/js/dataTables.min.js"></script>
  <script>
    $(document).ready(function() {{
      var table = $('#pairs').DataTable({{
        pageLength: 25,
        lengthMenu: [[10, 25, 50, 100, -1], [10, 25, 50, 100, "All"]],
        orderCellsTop: false,
        fixedHeader: true,
        initComplete: function() {{
          // per-column filter inputs in footer
          this.api().columns().every(function() {{
            var that = this;
            $('input', this.footer()).on('keyup change clear', function() {{
              if (that.search() !== this.value) {{
                that.search(this.value).draw();
              }}
            }});
          }});
        }}
      }});
    }});
  </script>
</body>
</html>
"""

OUT_PATH.write_text(HTML_TEMPLATE.format(
    n_rows=len(pairs_df),
    n_facts=pairs_df.fact_id.nunique(),
    thead=thead,
    tfoot=tfoot,
    tbody=tbody,
))

print(f"Wrote {OUT_PATH.resolve()}")
print(f"Open in browser:  file://{OUT_PATH.resolve()}")
print(f"Or:  xdg-open {OUT_PATH}")


Wrote /home/jay/repos/MIRROR-Eval/notebooks/pairs_browser.html
Open in browser:  file:///home/jay/repos/MIRROR-Eval/notebooks/pairs_browser.html
Or:  xdg-open pairs_browser.html


In [45]:
# Authenticate and set your dataset repo path.
# Opens an interactive login prompt — paste your HF token when asked.
login()

REPO_ID = "royal42/mta"
print("HF login successful")
print(f"Target repo: {REPO_ID}")
print(f"Rows to upload: {len(pairs_df)}")


HF login successful
Target repo: royal42/mta
Rows to upload: 3000


In [46]:
# Build a HF Dataset from pairs_df and push it to the Hub.
ds = Dataset.from_pandas(pairs_df, preserve_index=False)
print(ds)

ds.push_to_hub(REPO_ID)
print(f"\nUploaded dataset to: https://huggingface.co/datasets/{REPO_ID}")


Dataset({
    features: ['pair_id', 'fact_id', 'fact_type', 'fact', 'scenario_topic', 'axis_tone', 'axis_length', 'axis_urgency', 'axis_register', 'axis_domain', 'axis_request_type', 'axis_relevance', 'axis_opening', 'relevance_level', 'prompt_with_fact', 'prompt_without_fact'],
    num_rows: 3000
})


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Uploaded dataset to: https://huggingface.co/datasets/royal42/mta
